# House Price Prediction Project
## PRCP-1020-HousePricePred
### Project Overview

This project aims to:
1. Perform complete data analysis on house prices dataset
2. Build a machine learning model to predict house prices
3. Understand the relationship between house features and prices
4. Provide suggestions for customers looking to buy houses

The dataset contains 79 features describing residential homes in Ames, Iowa.

## Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load and Explore Data](#2-load-and-explore-data)
3. [Data Analysis Report (Task 1)](#3-data-analysis-report)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Machine Learning Models (Task 2)](#5-machine-learning-models)
6. [Model Comparison Report](#6-model-comparison-report)
7. [Customer Recommendations (Task 3)](#7-customer-recommendations)
8. [Challenges Faced Report](#8-challenges-faced-report)

---
## 1. Import Libraries

First, let's import all the libraries we'll need for this project.

---
## 2. Load and Explore Data

Let's load the dataset and take a first look at it.

In [ ]:
# Load the data
# Assuming the files are in a 'data' folder
df = pd.read_csv('data/data.csv')  # or whatever the actual filename is

print("Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Basic information about the dataset
print("Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary of Numerical Features:")
df.describe()

In [ ]:
# Check column names
print("Column names in the dataset:")
print(df.columns.tolist())

---
## 3. Data Analysis Report (Task 1)

### 3.1 Missing Values Analysis

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Percentage': missing_percent
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("Features with Missing Values:")
print(missing_df)

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(12, 6))
    plt.barh(missing_df.index[:20], missing_df['Percentage'][:20])
    plt.xlabel('Percentage of Missing Values')
    plt.ylabel('Features')
    plt.title('Top 20 Features with Missing Values')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

### 3.2 Target Variable Analysis (SalePrice)

In [ ]:
# Check if SalePrice exists in the dataset
if 'SalePrice' in df.columns:
    print("SalePrice Statistics:")
    print(df['SalePrice'].describe())
    print(f"\nMinimum Price: ${df['SalePrice'].min():,.2f}")
    print(f"Maximum Price: ${df['SalePrice'].max():,.2f}")
    print(f"Average Price: ${df['SalePrice'].mean():,.2f}")
    print(f"Median Price: ${df['SalePrice'].median():,.2f}")
else:
    print("SalePrice column not found. Please check the column name.")

In [ ]:
# Visualize SalePrice distribution
if 'SalePrice' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df['SalePrice'], bins=50, edgecolor='black')
    axes[0].set_xlabel('Sale Price')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of House Prices')
    
    # Box plot
    axes[1].boxplot(df['SalePrice'])
    axes[1].set_ylabel('Sale Price')
    axes[1].set_title('Box Plot of House Prices')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Check skewness of SalePrice
if 'SalePrice' in df.columns:
    from scipy.stats import skew
    skewness = skew(df['SalePrice'])
    print(f"Skewness of SalePrice: {skewness:.2f}")
    
    if skewness > 0.5:
        print("The distribution is positively skewed (right-skewed)")
    elif skewness < -0.5:
        print("The distribution is negatively skewed (left-skewed)")
    else:
        print("The distribution is approximately symmetric")

### 3.3 Numerical Features Analysis

In [ ]:
# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Number of Numerical Features: {len(numerical_cols)}")
print(f"Number of Categorical Features: {len(categorical_cols)}")
print(f"\nNumerical Features: {numerical_cols}")
print(f"\nCategorical Features: {categorical_cols}")

In [ ]:
# Correlation analysis with SalePrice
if 'SalePrice' in df.columns:
    correlations = df[numerical_cols].corr()['SalePrice'].sort_values(ascending=False)
    print("Top 10 features correlated with SalePrice:")
    print(correlations.head(11))  # 11 because SalePrice itself will be included

In [ ]:
# Visualize top correlations
if 'SalePrice' in df.columns:
    top_corr = correlations.head(11).iloc[1:]  # Exclude SalePrice itself
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(top_corr)), top_corr.values)
    plt.yticks(range(len(top_corr)), top_corr.index)
    plt.xlabel('Correlation with SalePrice')
    plt.title('Top 10 Features Correlated with Sale Price')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap of top features
if 'SalePrice' in df.columns:
    top_features = correlations.head(11).index.tolist()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap of Top Features')
    plt.tight_layout()
    plt.show()

### 3.4 Categorical Features Analysis

In [ ]:
# Analyze categorical features
print("Unique values in categorical features:")
for col in categorical_cols[:10]:  # Show first 10 for brevity
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
# Visualize some important categorical features vs SalePrice
if 'SalePrice' in df.columns:
    # Select a few important categorical features to visualize
    important_cat_features = ['Neighborhood', 'BldgType', 'HouseStyle']  # Example features
    
    # Check which features actually exist in the dataset
    available_features = [f for f in important_cat_features if f in df.columns]
    
    if len(available_features) > 0:
        fig, axes = plt.subplots(1, len(available_features), figsize=(18, 5))
        
        if len(available_features) == 1:
            axes = [axes]
        
        for idx, feature in enumerate(available_features):
            df.groupby(feature)['SalePrice'].median().sort_values().plot(kind='barh', ax=axes[idx])
            axes[idx].set_xlabel('Median Sale Price')
            axes[idx].set_title(f'Median Price by {feature}')
        
        plt.tight_layout()
        plt.show()

### 3.5 Outlier Detection

In [ ]:
# Detect outliers in important numerical features
important_num_features = ['GrLivArea', 'TotalBsmtSF', 'LotArea']  # Example features
available_num_features = [f for f in important_num_features if f in df.columns]

if len(available_num_features) > 0:
    fig, axes = plt.subplots(1, len(available_num_features), figsize=(18, 5))
    
    if len(available_num_features) == 1:
        axes = [axes]
    
    for idx, feature in enumerate(available_num_features):
        axes[idx].boxplot(df[feature].dropna())
        axes[idx].set_ylabel(feature)
        axes[idx].set_title(f'Box Plot - {feature}')
    
    plt.tight_layout()
    plt.show()

### 3.6 Data Analysis Summary

Based on the analysis above, we can summarize:

In [ ]:
print("="*80)
print("DATA ANALYSIS SUMMARY (TASK 1)")
print("="*80)
print(f"\n1. Dataset Overview:")
print(f"   - Total records: {df.shape[0]}")
print(f"   - Total features: {df.shape[1]}")
print(f"   - Numerical features: {len(numerical_cols)}")
print(f"   - Categorical features: {len(categorical_cols)}")

if 'SalePrice' in df.columns:
    print(f"\n2. Target Variable (SalePrice):")
    print(f"   - Average Price: ${df['SalePrice'].mean():,.2f}")
    print(f"   - Price Range: ${df['SalePrice'].min():,.2f} - ${df['SalePrice'].max():,.2f}")
    print(f"   - Median Price: ${df['SalePrice'].median():,.2f}")

print(f"\n3. Missing Values:")
if len(missing_df) > 0:
    print(f"   - Features with missing values: {len(missing_df)}")
    print(f"   - Most missing feature: {missing_df.index[0]} ({missing_df['Percentage'].iloc[0]:.2f}%)")
else:
    print("   - No missing values detected")

if 'SalePrice' in df.columns:
    print(f"\n4. Top 3 Correlated Features with Price:")
    top_3 = correlations.iloc[1:4]
    for i, (feature, corr) in enumerate(top_3.items(), 1):
        print(f"   {i}. {feature}: {corr:.3f}")

print("\n" + "="*80)

---
## 4. Data Preprocessing

Now we'll clean and prepare the data for machine learning.

### 4.1 Handle Missing Values

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

print(f"Original dataset shape: {df_processed.shape}")
print(f"Missing values before processing: {df_processed.isnull().sum().sum()}")

In [ ]:
# Handle missing values
# For numerical columns: fill with median
for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].median(), inplace=True)

# For categorical columns: fill with mode or 'None'
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].mode()[0] if len(df_processed[col].mode()) > 0 else 'None', inplace=True)

print(f"Missing values after processing: {df_processed.isnull().sum().sum()}")

### 4.2 Encode Categorical Variables

In [ ]:
# Encode categorical variables using Label Encoding
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le

print("Categorical variables encoded successfully!")
print(f"Dataset shape after encoding: {df_processed.shape}")

### 4.3 Feature Selection and Train-Test Split

In [ ]:
# Separate features and target
if 'SalePrice' in df_processed.columns:
    X = df_processed.drop('SalePrice', axis=1)
    y = df_processed['SalePrice']
    
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
else:
    print("SalePrice column not found. Cannot proceed with modeling.")

In [ ]:
# Split the data into training and testing sets
if 'SalePrice' in df_processed.columns:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Training set size: {X_train.shape[0]} samples")
    print(f"Testing set size: {X_test.shape[0]} samples")
    print(f"Training set percentage: {(X_train.shape[0] / len(df)) * 100:.1f}%")
    print(f"Testing set percentage: {(X_test.shape[0] / len(df)) * 100:.1f}%")

### 4.4 Feature Scaling

In [ ]:
# Scale the features
if 'SalePrice' in df_processed.columns:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("Features scaled successfully!")
    print(f"Scaled training data shape: {X_train_scaled.shape}")
    print(f"Scaled testing data shape: {X_test_scaled.shape}")

---
## 5. Machine Learning Models (Task 2)

Now we'll build and train multiple machine learning models to predict house prices.

### 5.1 Define Evaluation Function

In [ ]:
# Function to evaluate models
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a machine learning model
    """
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    # Print results
    print(f"\n{'='*60}")
    print(f"{model_name} Results")
    print(f"{'='*60}")
    print(f"Training R² Score: {train_r2:.4f}")
    print(f"Testing R² Score: {test_r2:.4f}")
    print(f"Training RMSE: ${train_rmse:,.2f}")
    print(f"Testing RMSE: ${test_rmse:,.2f}")
    print(f"Training MAE: ${train_mae:,.2f}")
    print(f"Testing MAE: ${test_mae:,.2f}")
    print(f"{'='*60}")
    
    return {
        'model_name': model_name,
        'model': model,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae
    }

print("Evaluation function defined!")

### 5.2 Train Multiple Models

In [ ]:
# Dictionary to store results
model_results = []

print("Starting model training...")
print("This might take a few minutes...")

#### 5.2.1 Linear Regression

In [ ]:
# Linear Regression
if 'SalePrice' in df_processed.columns:
    lr_model = LinearRegression()
    lr_results = evaluate_model(lr_model, X_train_scaled, X_test_scaled, y_train, y_test, "Linear Regression")
    model_results.append(lr_results)

#### 5.2.2 Ridge Regression

In [ ]:
# Ridge Regression
if 'SalePrice' in df_processed.columns:
    ridge_model = Ridge(alpha=1.0, random_state=42)
    ridge_results = evaluate_model(ridge_model, X_train_scaled, X_test_scaled, y_train, y_test, "Ridge Regression")
    model_results.append(ridge_results)

#### 5.2.3 Lasso Regression

In [ ]:
# Lasso Regression
if 'SalePrice' in df_processed.columns:
    lasso_model = Lasso(alpha=1.0, random_state=42)
    lasso_results = evaluate_model(lasso_model, X_train_scaled, X_test_scaled, y_train, y_test, "Lasso Regression")
    model_results.append(lasso_results)

#### 5.2.4 Decision Tree

In [ ]:
# Decision Tree
if 'SalePrice' in df_processed.columns:
    dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
    dt_results = evaluate_model(dt_model, X_train, X_test, y_train, y_test, "Decision Tree")
    model_results.append(dt_results)

#### 5.2.5 Random Forest

In [ ]:
# Random Forest
if 'SalePrice' in df_processed.columns:
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf_results = evaluate_model(rf_model, X_train, X_test, y_train, y_test, "Random Forest")
    model_results.append(rf_results)

#### 5.2.6 Gradient Boosting

In [ ]:
# Gradient Boosting
if 'SalePrice' in df_processed.columns:
    gb_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    gb_results = evaluate_model(gb_model, X_train, X_test, y_train, y_test, "Gradient Boosting")
    model_results.append(gb_results)

### 5.3 Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
if 'SalePrice' in df_processed.columns:
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 15 Most Important Features (from Random Forest):")
    print(feature_importance.head(15))
    
    # Visualize feature importance
    plt.figure(figsize=(12, 8))
    plt.barh(range(15), feature_importance['Importance'].head(15))
    plt.yticks(range(15), feature_importance['Feature'].head(15))
    plt.xlabel('Feature Importance')
    plt.title('Top 15 Most Important Features for House Price Prediction')
    plt.tight_layout()
    plt.show()

---
## 6. Model Comparison Report

Let's compare all models and determine the best one for production.

In [ ]:
# Create comparison dataframe
if len(model_results) > 0:
    comparison_df = pd.DataFrame(model_results)
    comparison_df = comparison_df[['model_name', 'train_r2', 'test_r2', 'train_rmse', 'test_rmse', 'train_mae', 'test_mae']]
    comparison_df = comparison_df.sort_values('test_r2', ascending=False)
    
    print("\n" + "="*80)
    print("MODEL COMPARISON REPORT")
    print("="*80)
    print("\n")
    print(comparison_df.to_string(index=False))
    print("\n" + "="*80)

In [ ]:
# Visualize model comparison
if len(model_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # R² Score comparison
    x_pos = np.arange(len(comparison_df))
    axes[0].bar(x_pos - 0.2, comparison_df['train_r2'], 0.4, label='Train R²')
    axes[0].bar(x_pos + 0.2, comparison_df['test_r2'], 0.4, label='Test R²')
    axes[0].set_xlabel('Models')
    axes[0].set_ylabel('R² Score')
    axes[0].set_title('Model Performance - R² Score')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(comparison_df['model_name'], rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # RMSE comparison
    axes[1].bar(x_pos - 0.2, comparison_df['train_rmse'], 0.4, label='Train RMSE')
    axes[1].bar(x_pos + 0.2, comparison_df['test_rmse'], 0.4, label='Test RMSE')
    axes[1].set_xlabel('Models')
    axes[1].set_ylabel('RMSE ($)')
    axes[1].set_title('Model Performance - RMSE')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(comparison_df['model_name'], rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Best model selection
if len(model_results) > 0:
    best_model_idx = comparison_df['test_r2'].idxmax()
    best_model_name = comparison_df.loc[best_model_idx, 'model_name']
    best_model_r2 = comparison_df.loc[best_model_idx, 'test_r2']
    best_model_rmse = comparison_df.loc[best_model_idx, 'test_rmse']
    
    print("\n" + "="*80)
    print("BEST MODEL RECOMMENDATION FOR PRODUCTION")
    print("="*80)
    print(f"\nRecommended Model: {best_model_name}")
    print(f"Test R² Score: {best_model_r2:.4f}")
    print(f"Test RMSE: ${best_model_rmse:,.2f}")
    print(f"\nJustification:")
    print(f"- Highest R² score on test data ({best_model_r2:.4f})")
    print(f"- Good balance between bias and variance")
    print(f"- Can explain {best_model_r2*100:.2f}% of variance in house prices")
    print("\n" + "="*80)

### 6.1 Prediction vs Actual Plot (Best Model)

In [ ]:
# Visualize predictions vs actual for best model
if len(model_results) > 0 and 'SalePrice' in df_processed.columns:
    # Get the best model
    best_model_obj = model_results[best_model_idx]['model']
    
    # Make predictions
    if best_model_name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        y_pred = best_model_obj.predict(X_test_scaled)
    else:
        y_pred = best_model_obj.predict(X_test)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual Price')
    plt.ylabel('Predicted Price')
    plt.title(f'Actual vs Predicted Prices - {best_model_name}')
    plt.tight_layout()
    plt.show()

---
## 7. Customer Recommendations (Task 3)

Based on our analysis, here are recommendations for customers looking to buy houses.

In [ ]:
# Function to provide recommendations
def get_house_recommendations(area=None, max_price=None, min_bedrooms=None):
    """
    Provide house recommendations based on customer requirements
    """
    recommendations = df.copy()
    
    # Filter by price if specified
    if max_price and 'SalePrice' in recommendations.columns:
        recommendations = recommendations[recommendations['SalePrice'] <= max_price]
    
    # Filter by area/neighborhood if specified
    if area and 'Neighborhood' in recommendations.columns:
        recommendations = recommendations[recommendations['Neighborhood'] == area]
    
    # Filter by bedrooms if specified
    if min_bedrooms and 'BedroomAbvGr' in recommendations.columns:
        recommendations = recommendations[recommendations['BedroomAbvGr'] >= min_bedrooms]
    
    return recommendations

print("Recommendation function created!")

In [ ]:
# General recommendations based on analysis
print("="*80)
print("CUSTOMER RECOMMENDATIONS (TASK 3)")
print("="*80)

if 'SalePrice' in df.columns:
    print("\n1. BUDGET-BASED RECOMMENDATIONS:")
    print(f"   - Budget-friendly (<$150,000): Look for houses with lower overall quality ratings")
    print(f"   - Mid-range ($150,000-$250,000): Average size homes with good condition")
    print(f"   - Premium (>$250,000): Larger homes with high quality materials and finishes")

print("\n2. KEY FACTORS THAT INCREASE HOUSE VALUE:")
if 'SalePrice' in df.columns and len(correlations) > 0:
    top_5_features = correlations.iloc[1:6]
    for i, (feature, corr) in enumerate(top_5_features.items(), 1):
        print(f"   {i}. {feature} (correlation: {corr:.3f})")

print("\n3. AREA-BASED RECOMMENDATIONS:")
if 'Neighborhood' in df.columns and 'SalePrice' in df.columns:
    top_neighborhoods = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).head(5)
    print("   Top 5 Neighborhoods by Median Price:")
    for i, (neighborhood, price) in enumerate(top_neighborhoods.items(), 1):
        print(f"   {i}. {neighborhood}: ${price:,.2f}")

print("\n4. BEST VALUE RECOMMENDATIONS:")
print("   - Look for recently remodeled houses (better condition at reasonable price)")
print("   - Houses with larger lot areas tend to have better resale value")
print("   - Good quality garages and basements add significant value")
print("   - Central air conditioning is a valuable feature")

print("\n5. THINGS TO AVOID:")
print("   - Houses with very old construction without recent remodeling")
print("   - Properties with poor overall condition ratings")
print("   - Houses in areas with lower median prices (unless budget-constrained)")

print("\n" + "="*80)

In [ ]:
# Example: Get recommendations for specific criteria
print("\nExample: Houses under $200,000 with at least 3 bedrooms")
if 'SalePrice' in df.columns:
    example_recs = get_house_recommendations(max_price=200000, min_bedrooms=3)
    print(f"Found {len(example_recs)} houses matching criteria")
    
    if len(example_recs) > 0:
        print(f"Average price: ${example_recs['SalePrice'].mean():,.2f}")
        print(f"Price range: ${example_recs['SalePrice'].min():,.2f} - ${example_recs['SalePrice'].max():,.2f}")

---
## 8. Challenges Faced Report

Let's document the challenges encountered during this project and how they were addressed.

In [ ]:
print("="*80)
print("CHALLENGES FACED REPORT")
print("="*80)

print("\n1. MISSING VALUES CHALLENGE")
print("   Problem: Multiple features had missing values")
if len(missing_df) > 0:
    print(f"   - Total features with missing values: {len(missing_df)}")
    print(f"   - Most affected feature: {missing_df.index[0]} ({missing_df['Percentage'].iloc[0]:.2f}% missing)")
print("   Solution:")
print("   - Numerical features: Filled with median values (robust to outliers)")
print("   - Categorical features: Filled with mode or 'None' category")
print("   Reasoning: Median is more robust than mean for numerical data with outliers")

print("\n2. CATEGORICAL ENCODING CHALLENGE")
print(f"   Problem: Dataset had {len(categorical_cols)} categorical features")
print("   Solution: Used Label Encoding to convert categories to numbers")
print("   Reasoning: Simple approach suitable for tree-based models")
print("   Alternative: Could use One-Hot Encoding for linear models (creates more features)")

if 'SalePrice' in df.columns:
    from scipy.stats import skew
    skewness = skew(df['SalePrice'].dropna())
    print("\n3. SKEWED TARGET VARIABLE CHALLENGE")
    print(f"   Problem: SalePrice distribution was skewed (skewness: {skewness:.2f})")
    print("   Potential Solution: Could apply log transformation to normalize distribution")
    print("   Current Approach: Used original values (tree-based models handle skewness well)")
    print("   Reasoning: Random Forest and Gradient Boosting don't require normal distribution")

print("\n4. FEATURE SELECTION CHALLENGE")
print(f"   Problem: Dataset has {df.shape[1]} features, some may be redundant")
print("   Solution: Used all features but analyzed feature importance")
print("   Reasoning: Let tree-based models automatically identify important features")
print("   Future Improvement: Could use Recursive Feature Elimination for optimization")

print("\n5. MODEL SELECTION CHALLENGE")
print("   Problem: Multiple algorithms available, needed to find the best one")
print("   Solution: Trained 6 different models and compared performance")
print("   Models tested: Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, Gradient Boosting")
print("   Reasoning: Ensemble methods (RF, GB) typically perform better for complex datasets")

print("\n6. OUTLIERS CHALLENGE")
print("   Problem: Some features had extreme values (outliers)")
print("   Solution: Kept outliers as they might represent luxury properties")
print("   Reasoning: Outliers contain valuable information about high-end market")
print("   Alternative: Could cap values at 95th/99th percentile if needed")

print("\n7. SCALING CHALLENGE")
print("   Problem: Features have different ranges (e.g., square feet vs number of rooms)")
print("   Solution: Applied StandardScaler for distance-based models")
print("   Reasoning: Linear models benefit from scaled features")
print("   Note: Tree-based models don't require scaling but we did it for consistency")

print("\n" + "="*80)

---
## 9. Conclusion

### Project Summary

In [ ]:
print("="*80)
print("PROJECT CONCLUSION")
print("="*80)

print("\n✓ TASKS COMPLETED:")
print("  1. Complete data analysis performed (EDA, missing values, correlations)")
print("  2. Multiple ML models built and evaluated")
print("  3. Relationships between features and prices identified")
print("  4. Customer recommendations provided based on analysis")
print("  5. Model comparison report created")
print("  6. Challenges documented with solutions")

if len(model_results) > 0:
    print(f"\n✓ BEST MODEL: {best_model_name}")
    print(f"  - Test R² Score: {best_model_r2:.4f}")
    print(f"  - Can explain {best_model_r2*100:.2f}% of price variation")
    print(f"  - Average prediction error: ${best_model_rmse:,.2f}")

print("\n✓ KEY INSIGHTS:")
if 'SalePrice' in df.columns and len(correlations) > 0:
    print(f"  - Most important factor: {correlations.index[1]}")
    print(f"  - Average house price: ${df['SalePrice'].mean():,.2f}")
    print(f"  - Price range: ${df['SalePrice'].min():,.2f} to ${df['SalePrice'].max():,.2f}")

print("\n✓ NEXT STEPS:")
print("  1. Fine-tune the best model with hyperparameter optimization")
print("  2. Try advanced ensemble methods (XGBoost, LightGBM)")
print("  3. Create feature engineering (interaction terms, polynomial features)")
print("  4. Deploy the model as a web application for real-time predictions")
print("  5. Gather user feedback and continuously improve the model")

print("\n" + "="*80)
print("Thank you for reviewing this project!")
print("="*80)

---
## Notes for Running This Notebook

### Before running:
1. Make sure the data files are in the correct location (update the path in cell under "2. Load and Explore Data")
2. Install required libraries if not already installed:
   ```bash
   pip install pandas numpy matplotlib seaborn scikit-learn scipy
   ```

### How to run:
1. Run cells sequentially from top to bottom
2. Each section builds on previous sections
3. Some cells may take a few minutes (especially model training)

### If you encounter errors:
1. Check that data file paths are correct
2. Verify column names match your dataset
3. Make sure all libraries are installed
4. Try restarting the kernel and running again

---
**End of Notebook**